# 03 - Train and tune the models

Compare preparation choices and tune Logistic Regression, Random Forest and Gradient Boosting using `data/processed/train.csv`, exported by notebook 02. All model choices use training data. The test set is reserved for notebook 04.

## 3.1 Load the training data and choose a score

Cross-validation (CV) checks a model on five held-aside parts of the training data. Each row is held aside once. Every experiment uses the same five groups.

Average precision (AP) measures how well purchases appear near the top of the model's ranking. Higher is better. A random ranking has AP around the purchase proportion. Compare the average AP across the five groups; standard deviation (SD) shows how much the scores vary.

A pipeline combines data preparation and the model. Each fold learns its category encoding and scaling from its own training rows.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

project_folder = Path.cwd()
pd.set_option('display.max_columns', 30)

# Keep one numerical thread so timing conditions are comparable across models.
import os
for setting in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS']:
    os.environ[setting] = '1'
from threadpoolctl import threadpool_limits
thread_limits = threadpool_limits(limits=1)
from time import perf_counter
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import average_precision_score, brier_score_loss, precision_score, recall_score

nominal_columns = ['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType']
skewed_columns = ['Administrative', 'Administrative_Duration', 'Informational',
                  'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration']
model_names = ['LogisticRegression', 'RandomForest', 'GradientBoosting']

In [2]:
training_data = pd.read_csv(project_folder / 'data' / 'processed' / 'train.csv',
                            float_precision='round_trip')
training_features = training_data.drop(columns=['Revenue', 'PageValues'])
training_target = training_data['Revenue']
validation_folds = list(StratifiedKFold(
    n_splits=5, shuffle=True, random_state=42).split(training_features, training_target))
print('Training sessions:', len(training_data))
print('Purchase proportion:', training_target.mean())

Training sessions: 9764
Purchase proportion: 0.15628840639082342


C:\Users\ongji\AppData\Local\Temp\ipykernel_9512\1286698480.py:1: Pandas4Warning: The 'float_precision' argument is deprecated. Use the default float precision instead.
  training_data = pd.read_csv(project_folder / 'data' / 'processed' / 'train.csv',


## 3.2 Preprocessing experiments
**Basic:** one-hot encode the six nominal columns, keep Weekend binary, scale numeric inputs for Logistic Regression only, and leave PageValues out.

**Log:** log1p means log(1 + value). It reduces the size of long count/duration tails and keeps zero as zero.

**Behaviour:** add total pages, total duration and product-page share. A session with no pages gets share zero.

**Weighting:** give purchases more influence during fitting. LR/RF use built-in class weights. For boosting we pass sample weights calculated separately inside each fold. No custom estimator class is needed.

In [3]:
def session_features(data, variant='basic'):
    result = data.copy()
    if variant == 'log':
        result[skewed_columns] = np.log1p(result[skewed_columns])
    if variant == 'behaviour':
        result['TotalPages'] = result[['Administrative', 'Informational', 'ProductRelated']].sum(axis=1)
        result['TotalDuration'] = result[['Administrative_Duration', 'Informational_Duration', 'ProductRelated_Duration']].sum(axis=1)
        # An empty session gets product-page share 0.
        result['ProductPageShare'] = np.divide(
            result['ProductRelated'], result['TotalPages'],
            out=np.zeros(len(result)), where=result['TotalPages'].to_numpy() != 0)
    return result

def make_preprocessor(scale_numeric=False, variant='basic', include_pagevalues=False):
    numeric_columns = skewed_columns + ['BounceRates', 'ExitRates', 'SpecialDay']
    if variant == 'behaviour':
        numeric_columns += ['TotalPages', 'TotalDuration', 'ProductPageShare']
    if include_pagevalues:
        numeric_columns += ['PageValues']
    numeric_step = StandardScaler() if scale_numeric else 'passthrough'
    return ColumnTransformer([
        ('nominal', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_columns),
        ('numeric', numeric_step, numeric_columns),
        ('binary', 'passthrough', ['Weekend'])], remainder='drop')

A small example shows what changes. These transformations work on each row separately; they do not learn anything from validation or test sessions.

In [4]:
example = training_features.iloc[:2].copy()
example.loc[example.index[0], ['Administrative', 'Informational', 'ProductRelated']] = 0
display(example[skewed_columns])
display(session_features(example, 'log')[skewed_columns])
display(session_features(example, 'behaviour')[['TotalPages', 'TotalDuration', 'ProductPageShare']])

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration
0,0,203.6,0,0.0,0,1052.400000
1,0,0.0,0,0.0,30,901.288889


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration
0,0.0,5.321057,0.0,0.0,0.000000,6.959778
1,0.0,0.000000,0.0,0.0,3.433987,6.804935


,TotalPages,TotalDuration,ProductPageShare
0,0,1256.000000,0.0
1,30,901.288889,1.0


### Run the same five-fold check
Out-of-fold predictions mean each training row is scored by a model fitted without that row. Keep these scores in memory while calculating AP, so the later weighting check can reuse them instead of fitting the same models again.

In [5]:
from sklearn.utils.class_weight import compute_sample_weight

def evaluate_folds(pipeline, features, balance_boosting=False):
    validation_ap = []
    out_of_fold_scores = np.zeros(len(features))
    for fit_rows, validation_rows in validation_folds:
        fold_model = clone(pipeline)
        fit_features = features.iloc[fit_rows]
        fit_target = training_target.iloc[fit_rows]
        if balance_boosting:
            # Calculate weights from this fold's training labels, never its validation labels.
            weights = compute_sample_weight('balanced', fit_target)
            # The double underscore sends weights to the pipeline step named 'model'.
            fold_model.fit(fit_features, fit_target, model__sample_weight=weights)
        else:
            fold_model.fit(fit_features, fit_target)
        validation_scores = fold_model.predict_proba(features.iloc[validation_rows])[:, 1]
        validation_ap.append(average_precision_score(training_target.iloc[validation_rows], validation_scores))
        out_of_fold_scores[validation_rows] = validation_scores
    summary = {
        'mean_ap': float(np.mean(validation_ap)),
        'sd_ap': np.std(validation_ap, ddof=1)}
    return summary, out_of_fold_scores

In [6]:
base_estimators = {
    'LogisticRegression': LogisticRegression(max_iter=2000, solver='lbfgs', random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42, n_jobs=1),
    'GradientBoosting': GradientBoostingClassifier(random_state=42)}
preprocessing_rows = []
experiment_scores = {}
for model_name in model_names:
    for variant in ['basic', 'log', 'behaviour', 'weighted']:
        print(model_name, variant, flush=True)
        estimator = clone(base_estimators[model_name])
        if variant == 'weighted' and model_name != 'GradientBoosting':
            estimator.set_params(class_weight='balanced')
        experiment_pipeline = Pipeline([
            ('features', FunctionTransformer(session_features, kw_args={'variant': variant})),
            ('preprocess', make_preprocessor(model_name == 'LogisticRegression', variant)),
            ('model', estimator)])
        balance_boosting = variant == 'weighted' and model_name == 'GradientBoosting'
        summary, scores = evaluate_folds(experiment_pipeline, training_features, balance_boosting)
        preprocessing_rows.append({'model': model_name, 'variant': variant, **summary})
        experiment_scores[(model_name, variant)] = scores

preprocessing_results = pd.DataFrame(preprocessing_rows)
selected_preprocessing = {}
for model_name in model_names:
    model_rows = preprocessing_results[preprocessing_results['model'] == model_name]
    basic_ap = model_rows.loc[model_rows['variant'] == 'basic', 'mean_ap'].iloc[0]
    preprocessing_results.loc[model_rows.index, 'delta_from_basic'] = model_rows['mean_ap'] - basic_ap
    best_row = model_rows.loc[model_rows['mean_ap'].idxmax()]
    # Keep one change only if it improves mean AP by at least 0.002.
    if best_row['mean_ap'] - basic_ap >= 0.002:
        selected_preprocessing[model_name] = best_row['variant']
    else:
        selected_preprocessing[model_name] = 'basic'
display(preprocessing_results[['model', 'variant', 'mean_ap', 'sd_ap', 'delta_from_basic']])
display(pd.Series(selected_preprocessing, name='Kept preprocessing'))

LogisticRegression basic


LogisticRegression log


LogisticRegression behaviour


LogisticRegression weighted


RandomForest basic


RandomForest log


... (4 outputs left out)

GradientBoosting behaviour


GradientBoosting weighted


,model,variant,mean_ap,sd_ap,delta_from_basic
0,LogisticRegression,basic,0.327941,0.029939,0.000000
1,LogisticRegression,log,0.338107,0.034613,0.010166
2,LogisticRegression,behaviour,0.328380,0.028962,0.000439
3,LogisticRegression,weighted,0.322841,0.029241,-0.005100
4,RandomForest,basic,0.361537,0.028552,0.000000
5,RandomForest,log,0.361860,0.027836,0.000323
6,RandomForest,behaviour,0.358294,0.020025,-0.003243
7,RandomForest,weighted,0.360115,0.023386,-0.001422
8,GradientBoosting,basic,0.367005,0.023303,0.000000
9,GradientBoosting,log,0.366694,0.023118,-0.000312


LogisticRegression      log
RandomForest          basic
GradientBoosting      basic
Name: Kept preprocessing, dtype: str

### PageValues sensitivity
Compare the same training rows, folds, untuned model settings and selected preprocessing with and without PageValues. A large AP difference shows that the feature is informative; it does not prove when it becomes available or establish leakage.

In [7]:
training_with_pagevalues = training_data.drop(columns='Revenue')
sensitivity_rows = []
for model_name in model_names:
    variant = selected_preprocessing[model_name]
    estimator = clone(base_estimators[model_name])
    if variant == 'weighted' and model_name != 'GradientBoosting':
        estimator.set_params(class_weight='balanced')
    sensitivity_pipeline = Pipeline([
        ('features', FunctionTransformer(session_features, kw_args={'variant': variant})),
        ('preprocess', make_preprocessor(model_name == 'LogisticRegression', variant, include_pagevalues=True)),
        ('model', estimator)])
    balance_boosting = variant == 'weighted' and model_name == 'GradientBoosting'
    summary, _ = evaluate_folds(sensitivity_pipeline, training_with_pagevalues, balance_boosting)
    without_pagevalues = preprocessing_results[
        (preprocessing_results['model'] == model_name) &
        (preprocessing_results['variant'] == variant)]['mean_ap'].iloc[0]
    row = {'model': model_name, 'variant': variant,
           'without_pagevalues_ap': without_pagevalues,
           'with_pagevalues_ap': summary['mean_ap'], 'with_pagevalues_sd': summary['sd_ap'],
           'delta_ap': summary['mean_ap'] - without_pagevalues}
    sensitivity_rows.append(row)
sensitivity_results = pd.DataFrame(sensitivity_rows)
display(sensitivity_results)

,model,variant,without_pagevalues_ap,with_pagevalues_ap,with_pagevalues_sd,delta_ap
0,LogisticRegression,log,0.338107,0.636733,0.024009,0.298627
1,RandomForest,basic,0.361537,0.734584,0.021611,0.373047
2,GradientBoosting,basic,0.367005,0.741876,0.024800,0.374871


### Try each set of model settings

A grid lists the settings to try. The helper below fits each candidate on the same five folds and returns the candidate with the highest mean validation AP. Each model's section shows its pipeline, settings and candidate scores.

The candidate table uses population SD; the winning summary uses sample SD. Both describe variation across the same five folds. Reusing these folds for preparation and tuning can make validation performance optimistic; there is no separate outer validation loop.

AP follows [scikit-learn's average precision definition](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html).

In [8]:
def run_grid_search(model_name, pipeline, parameter_grid):
    start_time = perf_counter()
    search = GridSearchCV(
        pipeline, parameter_grid, scoring='average_precision', cv=validation_folds,
        n_jobs=1, return_train_score=True, error_score='raise', refit=True)
    search.fit(training_features, training_target)
    tuning_seconds = perf_counter() - start_time
    candidates = pd.DataFrame(search.cv_results_)
    print('Every candidate: mean training/validation AP across the same five folds')
    candidate_scores = candidates[[
        'params', 'mean_train_score', 'std_train_score',
        'mean_test_score', 'std_test_score', 'rank_test_score']].rename(columns={
            'mean_train_score': 'mean_train_ap', 'std_train_score': 'sd_train_ap',
            'mean_test_score': 'mean_validation_ap', 'std_test_score': 'sd_validation_ap',
            'rank_test_score': 'validation_rank'})
    with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
        display(candidate_scores)
    best_candidate = candidates.iloc[search.best_index_]
    fold_scores = []
    for fold_number in range(5):
        fold_scores.append(best_candidate[f'split{fold_number}_test_score'])
    baseline = preprocessing_results[
        (preprocessing_results['model'] == model_name) &
        (preprocessing_results['variant'] == selected_preprocessing[model_name])].iloc[0]
    result = {
        'model': model_name, 'variant': selected_preprocessing[model_name],
        'default_ap': baseline['mean_ap'], 'tuned_ap': search.best_score_,
        'tuned_sd': float(np.std(fold_scores, ddof=1)),
        'delta_ap': search.best_score_ - baseline['mean_ap'],
        'params': search.best_params_, 'tuning_seconds': tuning_seconds}
    return result

## 3.3 Logistic Regression

This model gives each input a weight and combines the weighted inputs into a purchase score. Smaller C limits the size of the weights more strongly, which can reduce overfitting. Try C values of 0.1, 1 and 10.

Use the preparation selected above: log-transformed counts and durations, scaled numeric values, category indicators and binary Weekend. PageValues stays excluded.

In [9]:
lr_variant = selected_preprocessing['LogisticRegression']
lr_pipeline = Pipeline([
    ('features', FunctionTransformer(session_features, kw_args={'variant': lr_variant})),
    ('preprocess', make_preprocessor(scale_numeric=True, variant=lr_variant)),
    ('model', LogisticRegression(max_iter=2000, solver='lbfgs', random_state=42))])
lr_parameter_grid = {'model__C': [0.1, 1.0, 10.0]}
print('Preprocessing:', lr_variant)
display(lr_pipeline)
print('Parameters tested:', lr_parameter_grid)

Preprocessing: log


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('features', ...), ('preprocess', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function ses...001A12DF71EE0>
,"kw_args kw_args: dict, default=NoneDictionary of additional keyword arguments to pass to func... versionadded:: 0.18",{'variant': 'log'}
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_n

Parameters tested: {'model__C': [0.1, 1.0, 10.0]}


In [10]:
lr_result = run_grid_search('LogisticRegression', lr_pipeline, lr_parameter_grid)

Every candidate: mean training/validation AP across the same five folds


,params,mean_train_ap,sd_train_ap,mean_validation_ap,sd_validation_ap,validation_rank
0,{'model__C': 0.1},0.358554,0.007757,0.342172,0.031844,1
1,{'model__C': 1.0},0.361248,0.007348,0.338107,0.030959,2
2,{'model__C': 10.0},0.361247,0.007502,0.334372,0.029978,3


## 3.4 Random Forest

A random forest averages many decision trees. Each tree learns from a random sample of sessions and considers a subset of inputs when splitting. Combining trees can reduce the effect of one unstable tree.

Try 100 or 200 trees, minimum leaf sizes of 1 or 4, and two choices for the number of inputs considered at each split. Larger leaves make the fitted groups less specific. Use category indicators and unscaled numeric values; PageValues stays excluded.

In [11]:
rf_variant = selected_preprocessing['RandomForest']
rf_pipeline = Pipeline([
    ('features', FunctionTransformer(session_features, kw_args={'variant': rf_variant})),
    ('preprocess', make_preprocessor(scale_numeric=False, variant=rf_variant)),
    ('model', RandomForestClassifier(random_state=42, n_jobs=1))])
rf_parameter_grid = {'model__n_estimators': [100, 200], 'model__min_samples_leaf': [1, 4], 'model__max_features': ['sqrt', 0.5]}
print('Preprocessing:', rf_variant)
display(rf_pipeline)
print('Parameters tested:', rf_parameter_grid)

Preprocessing: basic


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('features', ...), ('preprocess', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function ses...001A12DF71EE0>
,"kw_args kw_args: dict, default=NoneDictionary of additional keyword arguments to pass to func... versionadded:: 0.18",{'variant': 'basic'}
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature

Parameters tested: {'model__n_estimators': [100, 200], 'model__min_samples_leaf': [1, 4], 'model__max_features': ['sqrt', 0.5]}


In [12]:
rf_result = run_grid_search('RandomForest', rf_pipeline, rf_parameter_grid)

Every candidate: mean training/validation AP across the same five folds


,params,mean_train_ap,sd_train_ap,mean_validation_ap,sd_validation_ap,validation_rank
0,"{'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__n_estimators': 100}",1.000000,7.021667e-17,0.361537,0.025537,6
1,"{'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__n_estimators': 200}",1.000000,1.719950e-16,0.368070,0.024131,4
2,"{'model__max_features': 'sqrt', 'model__min_samples_leaf': 4, 'model__n_estimators': 100}",0.865248,6.387796e-03,0.385786,0.025874,2
3,"{'model__max_features': 'sqrt', 'model__min_samples_leaf': 4, 'model__n_estimators': 200}",0.868747,5.546890e-03,0.386935,0.028561,1
4,"{'model__max_features': 0.5, 'model__min_samples_leaf': 1, 'model__n_estimators': 100}",1.000000,1.216188e-16,0.354173,0.020114,8
5,"{'model__max_features': 0.5, 'model__min_samples_leaf': 1, 'model__n_estimators': 200}",1.000000,8.599751e-17,0.356632,0.020716,7
6,"{'model__max_features': 0.5, 'model__min_samples_leaf': 4, 'model__n_estimators': 100}",0.966140,1.891175e-03,0.368039,0.025168,5
7,"{'model__max_features': 0.5, 'model__min_samples_leaf': 4, 'model__n_estimators': 200}",0.969911,1.812644e-03,0.371018,0.024114,3


## 3.5 Gradient Boosting

Gradient boosting builds small trees one after another. Each new tree helps correct the current model's errors. More trees, larger learning rates or deeper trees can learn more detail, but can also fit noise.

Try 100 or 200 stages, learning rates of 0.05 or 0.1, and depths of 2 or 3. Use category indicators and unscaled numeric values; PageValues stays excluded.

In [13]:
gb_variant = selected_preprocessing['GradientBoosting']
gb_pipeline = Pipeline([
    ('features', FunctionTransformer(session_features, kw_args={'variant': gb_variant})),
    ('preprocess', make_preprocessor(scale_numeric=False, variant=gb_variant)),
    ('model', GradientBoostingClassifier(random_state=42))])
gb_parameter_grid = {'model__n_estimators': [100, 200], 'model__learning_rate': [0.05, 0.1], 'model__max_depth': [2, 3]}
print('Preprocessing:', gb_variant)
display(gb_pipeline)
print('Parameters tested:', gb_parameter_grid)

Preprocessing: basic


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('features', ...), ('preprocess', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<function ses...001A12DF71EE0>
,"kw_args kw_args: dict, default=NoneDictionary of additional keyword arguments to pass to func... versionadded:: 0.18",{'variant': 'basic'}
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature

Parameters tested: {'model__n_estimators': [100, 200], 'model__learning_rate': [0.05, 0.1], 'model__max_depth': [2, 3]}


In [14]:
gb_result = run_grid_search('GradientBoosting', gb_pipeline, gb_parameter_grid)

Every candidate: mean training/validation AP across the same five folds


,params,mean_train_ap,sd_train_ap,mean_validation_ap,sd_validation_ap,validation_rank
0,"{'model__learning_rate': 0.05, 'model__max_depth': 2, 'model__n_estimators': 100}",0.395774,0.008006,0.362968,0.028947,7
1,"{'model__learning_rate': 0.05, 'model__max_depth': 2, 'model__n_estimators': 200}",0.429867,0.008057,0.364044,0.026984,5
2,"{'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 100}",0.455353,0.007434,0.365404,0.024965,4
3,"{'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__n_estimators': 200}",0.518335,0.006524,0.367314,0.024159,1
4,"{'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 100}",0.430812,0.007947,0.363885,0.028496,6
5,"{'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 200}",0.480703,0.007060,0.362467,0.026370,8
6,"{'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 100}",0.522182,0.010355,0.367005,0.020843,2
7,"{'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}",0.614292,0.006802,0.366359,0.020331,3


## 3.6 Compare the three models

Compare AP before and after tuning, the variation across folds, and the time spent tuning. Random Forest and Gradient Boosting have the highest two mean validation AP scores, so they form the pair evaluated in notebook 04. Random Forest is the strongest individual model by this training comparison. The fold scores vary, so the ranking is uncertain.

In [15]:
tuning_results = [lr_result, rf_result, gb_result]
tuning_comparison = pd.DataFrame(tuning_results)
display(tuning_comparison[['model', 'variant', 'default_ap', 'tuned_ap', 'tuned_sd', 'delta_ap', 'params', 'tuning_seconds']])
print('Manual ensemble: RandomForest + GradientBoosting')

,model,variant,default_ap,tuned_ap,tuned_sd,delta_ap,params,tuning_seconds
0,LogisticRegression,log,0.338107,0.342172,0.035603,0.004066,{'model__C': 0.1},2.399655
1,RandomForest,basic,0.361537,0.386935,0.031932,0.025399,"{'model__max_features': 'sqrt', 'model__min_sa...",177.926512
2,GradientBoosting,basic,0.367005,0.367314,0.027011,0.000309,"{'model__learning_rate': 0.05, 'model__max_dep...",154.519358


Manual ensemble: RandomForest + GradientBoosting


## 3.7 Extra check: weighting and probability scores
Reuse the out-of-fold scores from section 3.2. Compare the average purchase score with the observed purchase proportion. Brier score is mean squared probability error; lower is better. Weighting changes the probability scale and the errors at threshold 0.50. Neither final voting member retained weighting. The pooled OOF AP below is not the mean of the five fold APs, so it can differ from section 3.2.

In [16]:
weighting_rows = []
for model_name in model_names:
    for variant in ['basic', 'weighted']:
        scores = experiment_scores[(model_name, variant)]
        weighting_rows.append({
            'model': model_name, 'variant': variant,
            'purchase_prevalence': float(training_target.mean()),
            'mean_purchase_score': float(scores.mean()),
            'brier': float(brier_score_loss(training_target, scores)),
            'ap': float(average_precision_score(training_target, scores)),
            'precision_at_050': float(precision_score(training_target, scores >= 0.5, zero_division=0)),
            'recall_at_050': float(recall_score(training_target, scores >= 0.5))})
weighting_results = pd.DataFrame(weighting_rows)
display(weighting_results)

,model,variant,purchase_prevalence,mean_purchase_score,brier,ap,precision_at_050,recall_at_050
0,LogisticRegression,basic,0.156288,0.156395,0.119226,0.321292,0.475248,0.031455
1,LogisticRegression,weighted,0.156288,0.427262,0.209201,0.317768,0.266794,0.733945
2,RandomForest,basic,0.156288,0.161823,0.116250,0.355468,0.534351,0.091743
3,RandomForest,weighted,0.156288,0.258051,0.130842,0.354162,0.399174,0.316514
4,GradientBoosting,basic,0.156288,0.156086,0.115259,0.361712,0.547511,0.079292
5,GradientBoosting,weighted,0.156288,0.403245,0.190642,0.364168,0.286814,0.732634
